In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import requests
import zipfile
import io
from pathlib import Path

Matplotlib is building the font cache; this may take a moment.


In [12]:
base_dir = Path.cwd().parent  
# Path(__file__).resolve().parent in .py


downloading, extracting and savingn MEF income data

In [ ]:
# Only the year changes in the MEF download link, so we keep one template
# and fill in the year with .format(year=...). No long URL is repeated.
MEF_URL_TEMPLATE = (
    "https://www1.finanze.gov.it/finanze/analisi_stat/public/v_4_0_0/contenuti/"
    "Redditi_e_principali_variabili_IRPEF_su_base_comunale_CSV_{year}.zip?d=1615465800"
)

INCOME_YEARS = [2014, 2024]

mef_raw_data_dir = base_dir / "data" / "raw" / "mef"

# Create the 'data/raw/mef' directory if it doesn't already exist
mef_raw_data_dir.mkdir(parents=True, exist_ok=True)

# File types we accept from inside the ZIP (a tuple, because endswith() accepts tuples)
extensions = (".csv", ".xlsx", ".xls")

for year in INCOME_YEARS:
    url = MEF_URL_TEMPLATE.format(year=year)

    # Cache: if a file for this year is already on disk, skip the download.
    # mef_raw_data_dir / "..." is already a Path, so no need to wrap it in Path().
    if any((mef_raw_data_dir / f"mef_income_{year}{ext}").exists() for ext in extensions):
        print(f"mef_income_{year} already downloaded, skipping.")
        continue

    print(f"Downloading {year} from {url} ...")
    # timeout=30: give up after 30 seconds instead of freezing forever
    response = requests.get(url, timeout=30)
    # turn HTTP errors (404, 500, ...) into exceptions
    response.raise_for_status()

    # Open the ZIP directly from memory (no temporary .zip file on disk)
    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        # .lower() so that "DATA.CSV" is accepted too, not only "data.csv"
        data_files = [
            info for info in z.infolist()
            if info.filename.lower().endswith(extensions)
        ]

        # We save every file as "mef_income_<year>.<ext>". With two data files
        # in one ZIP, the second would overwrite the first, so we stop and say so.
        if len(data_files) != 1:
            raise ValueError(
                f"Expected 1 data file in the {year} ZIP, found {len(data_files)}: "
                f"{[info.filename for info in data_files]}"
            )

        fileinfo = data_files[0]
        extension = Path(fileinfo.filename).suffix.lower()
        target_path = mef_raw_data_dir / f"mef_income_{year}{extension}"

        # Read the file out of the ZIP and write it to disk
        target_path.write_bytes(z.read(fileinfo.filename))
        print(f"Extracted to: {target_path}")

Extracted to: /Users/armanfayyazi/Desktop/Folders/unimi/coding/project/housing-affordability-italy/data/raw/mef/mef_income_2014.csv
Extracted to: /Users/armanfayyazi/Desktop/Folders/unimi/coding/project/housing-affordability-italy/data/raw/mef/mef_income_2024.csv


In [30]:
mef_dfs = {}
for year in INCOME_YEARS:
    existing_file = next(
        (mef_raw_data_dir/f"mef_income_{year}{ext}" for ext in extensions
        if (mef_raw_data_dir/f"mef_income_{year}{ext}").exists()),
        None
    )

    if not existing_file:
        print(f"Warning: No data file found for {year} in {mef_raw_data_dir}")
        continue

    print(f"Loading {existing_file.name} into pandas...")

    if existing_file.suffix == ".csv":
        df = pd.read_csv(existing_file, delimiter=';',dtype={"Codice Istat Comune": str}, encoding="utf-8")
    else:
        df = pd.read_excel(existing_file)


    df['year'] = year

    mef_dfs[year] = df

Loading mef_income_2014.csv into pandas...
Loading mef_income_2024.csv into pandas...


In [31]:
df2014 = mef_dfs[2014]
df2014.head(10)

,Anno di imposta,Codice catastale,Codice Istat Comune,Denominazione Comune,Sigla Provincia,Regione,Codice Istat Regione,Numero contribuenti,Reddito da fabbricati - Frequenza,Reddito da fabbricati - Ammontare,...,Reddito complessivo da 15000 a 26000 euro - Ammontare,Reddito complessivo da 26000 a 55000 euro - Frequenza,Reddito complessivo da 26000 a 55000 euro - Ammontare,Reddito complessivo da 55000 a 75000 euro - Frequenza,Reddito complessivo da 55000 a 75000 euro - Ammontare,Reddito complessivo da 75000 a 120000 euro - Frequenza,Reddito complessivo da 75000 a 120000 euro - Ammontare,Reddito complessivo oltre 120000 euro - Frequenza,Reddito complessivo oltre 120000 euro - Ammontare,year
0,2014,A001,028001,ABANO TERME,PD,Veneto,5,14173,6977.0,11684674.0,...,98874364,2889.0,101381179.0,399.0,25555498.0,366.0,33642305.0,156.0,30814993.0,2014
1,2014,A004,098001,ABBADIA CERRETO,LO,Lombardia,3,212,89.0,70008.0,...,1669369,35.0,1128384.0,NaN,NaN,0.0,0.0,0.0,0.0,2014
2,2014,A005,097001,ABBADIA LARIANA,LC,Lombardia,3,2384,1187.0,1565868.0,...,17631595,584.0,19953253.0,60.0,3781730.0,43.0,3910776.0,20.0,4110091.0,2014
3,2014,A006,052001,ABBADIA SAN SALVATORE,SI,Toscana,9,4936,2602.0,3327596.0,...,34555118,718.0,24203466.0,56.0,3568341.0,29.0,2605800.0,11.0,1812778.0,2014
4,2014,A007,095001,ABBASANTA,OR,Sardegna,20,1755,853.0,777290.0,...,10090801,411.0,13455623.0,21.0,1363203.0,8.0,726624.0,NaN,NaN,2014
5,2014,A008,068001,ABBATEGGIO,PE,Abruzzo,13,289,122.0,75154.0,...,1570478,26.0,840483.0,NaN,NaN,0.0,0.0,0.0,0.0,2014
6,2014,A010,015002,ABBIATEGRASSO,MI,Lombardia,3,22731,11217.0,17402757.0,...,164513830,5561.0,191169405.0,645.0,40603241.0,408.0,37665279.0,155.0,31363478.0,2014
7,2014,A012,047001,ABETONE,PT,Toscana,9,533,223.0,395804.0,...,3295526,56.0,1756826.0,5.0,328710.0,NaN,NaN,NaN,NaN,2014
8,2014,A013,076001,ABRIOLA,PZ,Basilicata,17,1030,319.0,178759.0,...,4570426,56.0,1865787.0,5.0,334658.0,NaN,NaN,0.0,0.0,2014
9,2014,A014,088001,ACATE,RG,Sicilia,19,6834,2112.0,1554967.0,...,16917451,344.0,11310094.0,21.0,1330499.0,14.0,1329985.0,4.0,604104.0,2014


In [ ]:
df2014["Codice Istat Comune"].str.contains("Ã").sum()

# The result is zero. MEF names are uppercase ASCII; use ISTAT names for display

np.int64(0)

In [35]:
df2014.dtypes

Anno di imposta                                                                                                     int64
Codice catastale                                                                                                      str
Codice Istat Comune                                                                                                   str
Denominazione Comune                                                                                                  str
Sigla Provincia                                                                                                       str
Regione                                                                                                               str
Codice Istat Regione                                                                                                int64
Numero contribuenti                                                                                                 int64
Reddito da fabbricati - 

In [ ]:
CODE_COL = "Codice Istat Comune"
NAME_COL = "Denominazione Comune"
TAXPAYERS_COL = "Numero contribuenti"
TAXABLE_FREQ_COL = "Reddito imponibile - Frequenza"
TAXABLE_AMOUNT_COL = "Reddito imponibile - Ammontare"
MILAN_CODE = "015146"